# Standard Errors in Panel Data: A Beginner's Guide in Python

**Author:** Carlos Mendez | **Site:** [carlos-mendez.org](https://carlos-mendez.org/)

This notebook accompanies the blog post: [Standard Errors in Panel Data](https://carlos-mendez.org/post/python_panel_ses/).

We simulate a panel of 100 firms over 10 years with a known true effect ($\beta = 0.5$), then compare six standard error estimators to see how each affects inference.

**Inspired by:** [Gregoire (2024). Panel OLS Standard Errors](https://vincent.codes.finance/posts/panel-ols-standard-errors/)


## Setup

In [ ]:
# Install linearmodels if needed
!pip install -q linearmodels


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from linearmodels.panel import PanelOLS

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

# Dark theme palette
DARK_NAVY = "#0f1729"
GRID_LINE = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})


## 1. Data Generating Process

We simulate a panel with known structure so we can verify which SE estimators produce correct inference.

**True model:**

$$y_{it} = 2.0 + 0.5 \cdot x_{it} + \mu_i + \lambda_t + arepsilon_{it}$$

Where $\mu_i$ (firm fixed effect) is **correlated** with $x_{it}$ (R&D intensity), creating omitted variable bias in pooled regressions. The errors $arepsilon_{it}$ follow an AR(1) process within each firm ($ho = 0.5$).


In [ ]:
def simulate_panel(n_firms=100, n_years=10, seed=42):
    """Simulate a panel dataset with firm and time effects.
    True causal effect of x on y is beta = 0.5."""
    rng = np.random.default_rng(seed)
    firms = np.repeat(np.arange(1, n_firms + 1), n_years)
    years = np.tile(np.arange(2010, 2010 + n_years), n_firms)

    # Firm-level unobserved heterogeneity
    firm_ability = rng.normal(0, 2, n_firms)
    mu = np.repeat(firm_ability, n_years)

    # Time effects (business cycle)
    time_shocks = rng.normal(0, 0.5, n_years)
    lam = np.tile(time_shocks, n_firms)

    # R&D intensity (correlated with firm ability)
    x = 3.0 + 0.8 * mu + rng.normal(0, 1.5, n_firms * n_years)

    # Errors with within-firm AR(1) serial correlation
    eps = np.zeros(n_firms * n_years)
    rho_ar = 0.5
    for i in range(n_firms):
        start = i * n_years
        eps[start] = rng.normal(0, 1.5)
        for t in range(1, n_years):
            eps[start + t] = rho_ar * eps[start + t - 1] + rng.normal(0, 1.5)

    y = 2.0 + 0.5 * x + mu + lam + eps
    return pd.DataFrame({"firm": firms, "year": years, "y": y, "x": x})


df = simulate_panel(n_firms=100, n_years=10, seed=42)
print(f"Dataset shape: {df.shape}")
print(f"Number of firms: {df["firm"].nunique()}")
print(f"Number of years: {df["year"].nunique()}")
df.head()


In [ ]:
df.describe().round(4)


## 2. Exploring the Panel Structure

We decompose variance into **between-firm** (persistent differences across firms) and **within-firm** (year-to-year changes for a given firm) components.


In [ ]:
# Panel balance
obs_per_firm = df.groupby("firm").size()
print(f"Observations per firm: min={obs_per_firm.min()}, "
      f"max={obs_per_firm.max()}, mean={obs_per_firm.mean():.1f}")
print(f"Panel is {"balanced" if obs_per_firm.nunique() == 1 else "unbalanced"}")

# Between vs within variation
for var in ["y", "x"]:
    overall = df[var].std()
    between = df.groupby("firm")[var].mean().std()
    within = df.groupby("firm")[var].transform(lambda g: g - g.mean()).std()
    print(f"
Variation in {var}:")
    print(f"  Overall std:  {overall:.4f}")
    print(f"  Between std:  {between:.4f}")
    print(f"  Within std:   {within:.4f}")


In [ ]:
# Within-firm correlation
within_corr = (
    df.groupby("firm")
    .apply(lambda g: g["y"].corr(g["x"]), include_groups=False)
)
print(f"Within-firm correlation (y, x):")
print(f"  Mean:   {within_corr.mean():.4f}")
print(f"  Median: {within_corr.median():.4f}")


In [ ]:
# Figure: Panel structure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_linewidth(0)

rng_plot = np.random.default_rng(99)
sample_firms = sorted(rng_plot.choice(df["firm"].unique(), 10, replace=False))
colors_sample = [STEEL_BLUE, WARM_ORANGE, TEAL, "#e8956a", "#c4623d",
                 "#8fbfcc", "#e0a57a", "#5cc8c0", "#b0c4de", "#f0c8a0"]
for i, fid in enumerate(sample_firms):
    sub = df[df["firm"] == fid]
    axes[0].scatter(sub["x"], sub["y"], color=colors_sample[i % len(colors_sample)],
                    alpha=0.7, s=30, edgecolors=DARK_NAVY, linewidths=0.5)
axes[0].set_xlabel("R&D intensity (x)")
axes[0].set_ylabel("Firm performance (y)")
axes[0].set_title("10 sampled firms: x vs y", fontweight="bold")

axes[1].hist(within_corr, bins=20, color=STEEL_BLUE, edgecolor=DARK_NAVY, alpha=0.85)
axes[1].axvline(within_corr.mean(), color=WARM_ORANGE, linewidth=2,
                linestyle="--", label=f"Mean = {within_corr.mean():.2f}")
axes[1].set_xlabel("Within-firm correlation (y, x)")
axes[1].set_ylabel("Number of firms")
axes[1].set_title("Distribution of within-firm correlations", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.show()


## 3. Setting Up the MultiIndex

The  package requires a pandas MultiIndex with entity (firm) as the first level and time (year) as the second.


In [ ]:
df_panel = df.set_index(["firm", "year"])
print(f"MultiIndex levels: {df_panel.index.names}")
df_panel.head(3)


## 4. Pooled OLS --- the Naive Baseline

### Conventional standard errors
Pooled OLS ignores the panel structure entirely, treating all 1,000 observations as independent.


In [ ]:
mod_pooled = PanelOLS.from_formula("y ~ 1 + x", data=df_panel)
res_pooled = mod_pooled.fit(cov_type="unadjusted")

beta_pooled = res_pooled.params["x"]
se_pooled = res_pooled.std_errors["x"]
t_pooled = res_pooled.tstats["x"]
print(f"Coefficient on x: {beta_pooled:.4f}  (true = 0.5)")
print(f"Conventional SE:  {se_pooled:.4f}")
print(f"t-statistic:      {t_pooled:.4f}")


The coefficient is **1.03** --- more than double the true 0.5. This is omitted variable bias: high-ability firms invest more in R&D *and* perform better.

### White (heteroskedasticity-robust) standard errors


In [ ]:
res_white = mod_pooled.fit(cov_type="robust")
se_white = res_white.std_errors["x"]
t_white = res_white.tstats["x"]
print(f"White SE:    {se_white:.4f}")
print(f"t-statistic: {t_white:.4f}")


## 5. Clustered Standard Errors

Clustering allows arbitrary correlation among observations within the same group (firm or year).


In [ ]:
# Entity-clustered
res_cl_entity = mod_pooled.fit(cov_type="clustered", cluster_entity=True)
se_cl_entity = res_cl_entity.std_errors["x"]
print(f"Entity-clustered SE: {se_cl_entity:.4f}  (t = {res_cl_entity.tstats["x"]:.4f})")

# Time-clustered
res_cl_time = mod_pooled.fit(cov_type="clustered", cluster_time=True)
se_cl_time = res_cl_time.std_errors["x"]
print(f"Time-clustered SE:   {se_cl_time:.4f}  (t = {res_cl_time.tstats["x"]:.4f})")

# Two-way clustered
res_cl_both = mod_pooled.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
se_cl_both = res_cl_both.std_errors["x"]
print(f"Two-way clustered SE: {se_cl_both:.4f}  (t = {res_cl_both.tstats["x"]:.4f})")


## 6. Entity Fixed Effects + Clustered SEs

Fixed effects remove all time-invariant firm-level heterogeneity by demeaning each firm's observations. This eliminates the omitted variable bias.


In [ ]:
# Entity FE
mod_fe = PanelOLS.from_formula("y ~ 1 + x + EntityEffects", data=df_panel)
res_fe_cl = mod_fe.fit(cov_type="clustered", cluster_entity=True)
beta_fe = res_fe_cl.params["x"]
se_fe_cl = res_fe_cl.std_errors["x"]
print(f"FE coefficient on x:    {beta_fe:.4f}  (true = 0.5)")
print(f"Entity-clustered SE:    {se_fe_cl:.4f}")
print(f"t-statistic:            {res_fe_cl.tstats["x"]:.4f}")

# Two-way FE
mod_twfe = PanelOLS.from_formula("y ~ 1 + x + EntityEffects + TimeEffects", data=df_panel)
res_twfe = mod_twfe.fit(cov_type="clustered", cluster_entity=True)
beta_twfe = res_twfe.params["x"]
se_twfe = res_twfe.std_errors["x"]
print(f"
TWFE coefficient on x:  {beta_twfe:.4f}")
print(f"Entity-clustered SE:    {se_twfe:.4f}")
print(f"t-statistic:            {res_twfe.tstats["x"]:.4f}")


## 7. Driscoll-Kraay Standard Errors

Driscoll-Kraay SEs account for both cross-sectional and temporal dependence using a kernel-based approach.


In [ ]:
res_dk = mod_pooled.fit(cov_type="kernel", kernel="bartlett", bandwidth=3)
se_dk = res_dk.std_errors["x"]
print(f"Driscoll-Kraay SE (BW=3): {se_dk:.4f}")
print(f"t-statistic:              {res_dk.tstats["x"]:.4f}")


## 8. Full Comparison

### Summary table


In [ ]:
results = {
    "Pooled OLS (conventional)": (beta_pooled, se_pooled, res_pooled.tstats["x"]),
    "Pooled OLS (White/HC)": (beta_pooled, se_white, res_white.tstats["x"]),
    "Pooled OLS (cluster: entity)": (beta_pooled, se_cl_entity, res_cl_entity.tstats["x"]),
    "Pooled OLS (cluster: time)": (beta_pooled, se_cl_time, res_cl_time.tstats["x"]),
    "Pooled OLS (cluster: both)": (beta_pooled, se_cl_both, res_cl_both.tstats["x"]),
    "Entity FE (cluster: entity)": (beta_fe, se_fe_cl, res_fe_cl.tstats["x"]),
    "Two-way FE (cluster: entity)": (beta_twfe, se_twfe, res_twfe.tstats["x"]),
    "Pooled OLS (Driscoll-Kraay)": (beta_pooled, se_dk, res_dk.tstats["x"]),
}

summary_df = pd.DataFrame(
    [(name, b, se, t, abs(t) > 1.96)
     for name, (b, se, t) in results.items()],
    columns=["Model / SE Type", "Coefficient", "Std. Error", "t-stat", "Reject H0 (5%)"]
)
summary_df


### SE comparison and confidence intervals

In [ ]:
# SE comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)
se_names = list(results.keys())
se_values = [results[k][1] for k in se_names]
colors_bar = [STEEL_BLUE if "FE" not in k else WARM_ORANGE for k in se_names]
bars = ax.barh(range(len(se_names)), se_values, color=colors_bar, edgecolor=DARK_NAVY, height=0.65)
ax.set_yticks(range(len(se_names)))
ax.set_yticklabels(se_names, fontsize=10)
ax.set_xlabel("Standard error of coefficient on x")
ax.set_title("How SE estimates vary by method", fontweight="bold", pad=15)
for bar, val in zip(bars, se_values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=10, color=LIGHT_TEXT)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Confidence intervals
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)
for i, (name, (b, se, t)) in enumerate(results.items()):
    ci_lo, ci_hi = b - 1.96 * se, b + 1.96 * se
    color = STEEL_BLUE if "FE" not in name else WARM_ORANGE
    ax.plot([ci_lo, ci_hi], [i, i], color=color, linewidth=2.5, solid_capstyle="round")
    ax.scatter([b], [i], color=color, s=60, zorder=5, edgecolors=DARK_NAVY)
ax.axvline(0.5, color=TEAL, linewidth=1.5, linestyle="--", alpha=0.7, label="True beta = 0.5")
ax.set_yticks(range(len(results)))
ax.set_yticklabels(list(results.keys()), fontsize=10)
ax.set_xlabel("Coefficient on x (with 95% CI)")
ax.set_title("95% confidence intervals across SE methods", fontweight="bold", pad=15)
ax.legend(loc="upper left")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Monte Carlo Simulation

We run 500 simulations using Entity FE models (which are unbiased) to test whether different SE estimators produce the correct 5% rejection rate when the null hypothesis is true.


In [ ]:
N_SIM = 500
reject_counts = {
    "FE + conventional": 0,
    "FE + White (HC)": 0,
    "FE + cluster: entity": 0,
    "FE + cluster: time": 0,
    "FE + cluster: both": 0,
    "TWFE + cluster: entity": 0,
}

print(f"Running {N_SIM} simulations...")
for sim in range(N_SIM):
    sim_df = simulate_panel(n_firms=100, n_years=10, seed=sim + 1000)
    sim_panel = sim_df.set_index(["firm", "year"])
    mod_fe_sim = PanelOLS.from_formula("y ~ 1 + x + EntityEffects", data=sim_panel)

    for name, kwargs in [("FE + conventional", {"cov_type": "unadjusted"}),
                          ("FE + White (HC)", {"cov_type": "robust"}),
                          ("FE + cluster: entity", {"cov_type": "clustered", "cluster_entity": True}),
                          ("FE + cluster: time", {"cov_type": "clustered", "cluster_time": True}),
                          ("FE + cluster: both", {"cov_type": "clustered", "cluster_entity": True, "cluster_time": True})]:
        r = mod_fe_sim.fit(**kwargs)
        if abs((r.params["x"] - 0.5) / r.std_errors["x"]) > 1.96:
            reject_counts[name] += 1

    mod_twfe_sim = PanelOLS.from_formula("y ~ 1 + x + EntityEffects + TimeEffects", data=sim_panel)
    r = mod_twfe_sim.fit(cov_type="clustered", cluster_entity=True)
    if abs((r.params["x"] - 0.5) / r.std_errors["x"]) > 1.96:
        reject_counts["TWFE + cluster: entity"] += 1

rejection_rates = {k: v / N_SIM for k, v in reject_counts.items()}
print("
Empirical rejection rates at 5% level (H0: beta=0.5 is true):")
for method, rate in rejection_rates.items():
    print(f"  {method:30s}: {rate:.3f} ({reject_counts[method]}/{N_SIM})")


In [ ]:
# Monte Carlo rejection rate figure
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_linewidth(0)
mc_names = list(rejection_rates.keys())
mc_rates = [rejection_rates[k] for k in mc_names]
mc_colors = [WARM_ORANGE if r > 0.10 else TEAL if 0.03 <= r <= 0.08 else STEEL_BLUE for r in mc_rates]
bars = ax.barh(range(len(mc_names)), mc_rates, color=mc_colors, edgecolor=DARK_NAVY, height=0.6)
ax.axvline(0.05, color=TEAL, linewidth=2, linestyle="--", label="Nominal 5% level")
ax.set_yticks(range(len(mc_names)))
ax.set_yticklabels(mc_names, fontsize=11)
ax.set_xlabel("Empirical rejection rate")
ax.set_title(f"Monte Carlo rejection rates ({N_SIM} simulations, H0 true)", fontweight="bold", pad=15)
ax.legend(loc="lower right")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
for bar, val in zip(bars, mc_rates):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.1%}", va="center", fontsize=10, color=LIGHT_TEXT)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## Key Takeaways

1. **Standard errors cannot fix bias.** Pooled OLS overestimated the R&D effect at 1.03 (true: 0.5) regardless of SE choice. Entity FE recovered ~0.48.
2. **Clustering dimension matters.** Entity-clustered SEs (0.062) were 80% larger than conventional SEs (0.035). Time-clustered SEs (0.017) were misleadingly small with only 10 year-clusters.
3. **Monte Carlo validation is essential.** Entity-clustered SEs on FE rejected at ~6.6% (close to 5%). Time-clustered SEs rejected at ~9% --- nearly double the nominal rate.
4. **FE + entity-clustered is the reliable default.** It addresses both bias (via FE) and inference (via clustering).

## References

1. [Gregoire, V. (2024). Panel OLS Standard Errors. *Vincent Codes Finance*.](https://vincent.codes.finance/posts/panel-ols-standard-errors/)
2. [linearmodels --- Kevin Sheppard.](https://bashtage.github.io/linearmodels/panel/index.html)
3. [Petersen, M. A. (2009). Estimating Standard Errors in Finance Panel Data Sets. *Review of Financial Studies*, 22(1), 435-480.](https://doi.org/10.1093/rfs/hhn053)
4. [Cameron, Gelbach & Miller (2011). Robust Inference with Multiway Clustering. *JBES*, 29(2), 238-249.](https://doi.org/10.1198/jbes.2010.07136)
5. [Driscoll & Kraay (1998). Consistent Covariance Matrix Estimation. *Review of Economics and Statistics*.](https://doi.org/10.1162/003465398557549)
